# State Spending: Where the Money Goes
Expenditures by Department of Finance agency, collapsed to a stable set of areas (agencies were reorganized several times since 1984-85). General obligation bond debt service is split out into its own area. 'State-funded' means General Fund plus special funds; the 'all funds' view adds bond and federal money.

In [1]:
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from ca_taxes.io import PROCESSED
from ca_taxes.deflate import add_views, base_fy, LAST_ACTUAL_FY
from ca_taxes.style import stacked_area_toggle, base_layout, PALETTE, SOURCE_NOTE
pio.renderers.default = "plotly_mimetype"
exp = pd.read_csv(PROCESSED / "state_expenditures.csv")
state = exp[exp.fund.isin(["General Fund", "Special Funds"])]
d = add_views(state.groupby(["fy", "area"], as_index=False).amount_k.sum())
stacked_area_toggle(d, "area", "State-funded spending by area", "General Fund + special funds. " + SOURCE_NOTE).show()

In [2]:
gfx = add_views(exp[exp.fund == "General Fund"].groupby(["fy", "area"], as_index=False).amount_k.sum())
stacked_area_toggle(gfx, "area", "General Fund spending by area", SOURCE_NOTE).show()

In [3]:
allf = add_views(exp.groupby(["fy", "area"], as_index=False).amount_k.sum())
stacked_area_toggle(allf, "area", "Spending by area, all funds", "Includes bond and federal funds. " + SOURCE_NOTE).show()

## Which departments grew fastest?
Top 12 departments by cumulative real state-funded spending, in real dollars per person.

In [4]:
dep = add_views(state.groupby(["fy", "department"], as_index=False).amount_k.sum())
top = dep.groupby("department").real_b.sum().sort_values(ascending=False).head(12).index
w = dep[dep.department.isin(top)].pivot_table(index="fy", columns="department", values="real_pc")
fig = go.Figure()
for i, c in enumerate(top):
    fig.add_trace(go.Scatter(x=w.index, y=w[c], name=c, mode="lines", line=dict(color=PALETTE[i % len(PALETTE)], width=2)))
fig.update_layout(yaxis_title=f"{base_fy()} $ per person", xaxis_title="Fiscal year (ending June 30)", yaxis_tickformat="$,.0f")
base_layout(fig, "Largest departments: real state-funded spending per person", SOURCE_NOTE, height=620)
fig.show()